## Entendimiento del problema

### By: Carlos Javier Palacios Sanchez

### Date: 2026-08-27

### Description:

General description of the notebook

# Requerimiento

Crear una nueva rama de git (Usar [Gitflow](https://joserzapata.github.io/courses/ciencia-datos-en-produccion/control-versiones/branching-model/)) y Crear un notebook  para la obtención de datos tipo RAW.

- crear la rama a partir de un issue: <https://docs.github.com/es/issues/tracking-your-work-with-issues/using-issues/creating-a-branch-for-an-issue>

- Realizar los pasos de: <https://joserzapata.github.io/post/ciencia-datos-proyecto-python/1-data/>

Descargar los datos y entender el problema a realizar, contestar en el notebook

- ¿Cual es el objetivo del problema?
- ¿Cómo se usará su solución?
- ¿Cuáles son las soluciones actuales (si las hay)?
- ¿Cómo se debe enmarcar este problema (supervisado / no supervisado, en línea / fuera de línea, etc.)
- ¿Cómo se debe medir el desempeño o el rendimiento de la solución, una primera intuicion?
- ¿La medida de desempeño está alineada con el objetivo del problema?
- ¿Cuál sería el desempeño o rendimiento mínimo necesario para alcanzar el objetivo del problema?
- ¿Cuáles son los problemas parecidos? ¿Se puede reutilizar experiencias o herramientas ya creadas?
- ¿Hay experiencia del problema disponible?
- (Importante) ¿Cómo se puede resolver el problema manualmente?
- Hacer un listado de los supuestos que hay hasta este momento.
- Cual es la fuente de los datos?
- Como se actualizan los datos?
- Cada cuanto tiempo se actualizan los datos

# Entregables

Notebook para obtención de los datos.

Se debe realizar un Pull request para ingresar el notebook a la rama`main` para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD


## 📊 Analisis

1. ¿Cuál es el objetivo del problema?
Predecir, a partir de variables clínicas obtenidas mediante procedimientos no invasivos,
si un paciente presenta enfermedad arterial coronaria significativa —definida en la
fuente original como un estrechamiento superior al 50 % del diámetro en al menos una
arteria coronaria principal, verificado por angiografía—.
El objetivo operativo no es reemplazar el diagnóstico angiográfico, sino estimar la
probabilidad de enfermedad antes de someter al paciente a un procedimiento
invasivo, de modo que la angiografía se reserve para quienes tienen probabilidad pretest
elevada. En términos de negocio esto se traduce en dos beneficios simultáneos:
reducir procedimientos invasivos innecesarios (con su costo y su riesgo asociado) y
disminuir los falsos negativos, es decir, los pacientes con enfermedad que quedan sin
estudiar.
La variable objetivo disease es binaria: 0 (ausencia de enfermedad significativa) y 1
(presencia). En el archivo entregado, sobre los 2.919 registros con etiqueta válida la
distribución es 53,99 % / 46,01 %, es decir, un problema prácticamente balanceado,
lo cual simplifica el tratamiento posterior.

2. ¿Cómo se usará su solución?
La solución se concibe como un sistema de apoyo a la decisión clínica (CDSS), no
como un sistema de decisión automática. El uso previsto es el siguiente:
Punto de aplicación: consulta de cardiología o medicina interna, en el momento en
que ya se dispone de anamnesis, electrocardiograma en reposo, prueba de esfuerzo
y perfil lipídico —todas las variables del modelo provienen de esos estudios y
ninguna requiere angiografía—.
Entrada: las 13 variables predictoras del paciente, capturadas en un formulario o
extraídas de la historia clínica electrónica.
Salida: una probabilidad de enfermedad coronaria acompañada de una
estratificación en riesgo bajo / intermedio / alto y de los factores que más
contribuyeron a esa estimación.
Acción que desencadena: priorización para angiografía en el grupo de riesgo alto,
y seguimiento o manejo médico en el de riesgo bajo. La decisión final permanece en
el médico tratante.
Modalidad de consumo: inferencia individual bajo demanda (una petición por
paciente), no procesamiento masivo.
Esta definición de uso condiciona dos requisitos no funcionales: el modelo debe ser
interpretable —un clínico no adoptará una recomendación que no puede justificar
ante el paciente— y la latencia debe ser de segundos, no de minutos.

3. ¿Cuáles son las soluciones actuales (si las hay)?
El problema no carece de solución: existe una práctica clínica establecida contra la cual
todo modelo debe compararse.
Angiografía coronaria (estándar de oro). Es el procedimiento con el que se
construyó la etiqueta del dataset. Es concluyente, pero invasivo, costoso y con riesgo
de complicaciones; no puede aplicarse a toda la población sintomática.
Juicio clínico del especialista. El cardiólogo integra mentalmente los mismos
factores que contiene el dataset —tipo de dolor torácico, edad, sexo, respuesta a la
prueba de esfuerzo— para decidir a quién derivar. Es el benchmark real de la
solución propuesta.
Escalas y modelos de probabilidad pre-test. Instrumentos como la tabla de
Diamond–Forrester y sus actualizaciones estiman la probabilidad de enfermedad
coronaria a partir de edad, sexo y tipo de dolor torácico. Son transparentes y
ampliamente validados, pero usan pocas variables.
Escalas de riesgo cardiovascular poblacional (Framingham, SCORE, ASCVD).
Conviene señalar que estas responden a una pregunta distinta —riesgo de evento a
10 años— y no a la presencia actual de estenosis, por lo que no son sustitutos
directos.
Pruebas no invasivas complementarias: ecocardiografía de estrés, SPECT de
perfusión miocárdica, angiotomografía coronaria. Mejoran la discriminación pero
implican costo y disponibilidad limitada.
El nicho de la solución propuesta es, por tanto, ordenar la derivación entre el juicio
clínico y la angiografía, usando información ya disponible y sin costo marginal de
adquisición.

4. ¿Cómo se debe enmarcar este problema?
Dimensión Encuadre Justificación
Tipo de
aprendizaje
Supervisado Se dispone de la etiqueta disease verificada
angiográficamente para cada registro.
Tarea Clasificación
binaria con salida
probabilística
Dos clases mutuamente excluyentes. Se prefiere la
probabilidad calibrada sobre la etiqueta dura, porque permite
mover el umbral según el costo clínico.
Entrenamiento Fuera de línea
(batch)
El conjunto es estático y de tamaño reducido; no llegan datos
nuevos de forma continua. El reentrenamiento sería un
evento programado, no incremental.
Inferencia En línea, por
instancia
La predicción se solicita en el momento de la consulta, un
paciente a la vez.
Naturaleza de
los datos
Tabular, mixta Variables numéricas continuas ( age , rest_bp , chol ,
max_hr , old_peak ), ordinales ( slope , ca ) y categóricas
nominales ( chest_pain , thal , rest_ecg , sex ).
Volumen Muy pequeño Tras eliminar la replicación del archivo quedan 297 registros
únicos completos. Esto descarta arquitecturas de alta
capacidad y obliga a validación cruzada en lugar de una
simple partición hold-out.
Es un problema de aprendizaje supervisado clásico, limitado por el tamaño muestral
y no por la complejidad del algoritmo. La conclusión práctica es que el esfuerzo
debe concentrarse en la ingeniería de características, la validación honesta y la
calibración, y no en la búsqueda de modelos sofisticados.

5. ¿Cómo se debe medir el desempeño? Una primera intuición
La exactitud (accuracy) es insuficiente porque trata por igual dos errores de
consecuencias muy distintas: no detectar a un paciente enfermo puede costarle la vida;
señalar como sospechoso a un paciente sano genera una angiografía innecesaria. La
métrica debe reflejar esa asimetría.
Se propone la siguiente jerarquía:
Métrica principal — Sensibilidad (recall de la clase positiva). Proporción de
pacientes con enfermedad que el modelo identifica. Es la métrica que protege contra
el error costoso.
Métrica de restricción — Especificidad / valor predictivo positivo. Se maximiza
la sensibilidad sujeta a mantener la especificidad en un nivel que evite saturar el
servicio de hemodinamia.
Métrica de discriminación global — AUC-ROC. Independiente del umbral,
permite comparar modelos entre sí durante el desarrollo. Complementada con AUCPR
dado que ambas clases importan.
Calibración — Brier score y curva de calibración. Si la salida se presenta como
«68 % de probabilidad», ese número debe ser verídico; de lo contrario el clínico lo
interpretará mal.
Métrica resumen sugerida: sensibilidad al umbral operativo, reportada junto con
especificidad y AUC. Un valor F2 (que pondera la sensibilidad al doble que la precisión) es
una alternativa razonable como métrica única de optimización.

6. ¿La medida de desempeño está alineada con el objetivo del
problema?
Sí, con una precisión necesaria. El objetivo declarado en la sección 1 es doble: no dejar
pasar pacientes enfermos y no derivar innecesariamente a los sanos. Una métrica única
no captura ese equilibrio, y por eso se adoptó una principal con una restricción.
La alineación puede verificarse por contraejemplo. Con la distribución de clases
observada (53,99 % negativos), un clasificador trivial que declare «sano» a todos
alcanza 53,99 % de exactitud y una sensibilidad de 0 %: sería un modelo inútil y
potencialmente dañino que, sin embargo, luce aceptable bajo la métrica equivocada. La
sensibilidad lo descarta de inmediato. Ese es exactamente el comportamiento que se
espera de una métrica bien alineada.
Queda una advertencia honesta: la alineación es parcial. Ninguna de estas métricas
mide el desenlace que en última instancia importa —mortalidad evitada, eventos
cardiovasculares prevenidos, angiografías ahorradas—. La métrica estadística es un sustituto del objetivo clínico, y la validación definitiva de la utilidad del sistema exigiría
un estudio prospectivo, no una tabla de resultados sobre datos históricos.

7. ¿Cuál sería el desempeño mínimo necesario?
El umbral mínimo se fija por referencia a las alternativas existentes, no de manera
arbitraria. Se establecen tres niveles:
Nivel Criterio Valor
Piso absoluto
(rechazo)
Exactitud del clasificador de clase mayoritaria. Por debajo de este valor
el modelo no aporta nada.
53,99 %
Mínimo aceptable Sensibilidad en el umbral operativo. Por debajo, el sistema deja escapar
más de 1 de cada 5 pacientes enfermos y no es defendible
clínicamente.
≥ 80 %
Mínimo aceptable Especificidad simultánea. Garantiza que la reducción de falsos
negativos no se pague con una avalancha de derivaciones.
≥ 70 %
Objetivo
deseable
AUC-ROC en validación cruzada, coherente con lo publicado sobre este
conjunto de datos.
≥ 0,88
Condición de
superioridad
El modelo debe superar de forma consistente a la regla heurística
manual de la sección 10, que ya alcanza cerca del 84 % de exactitud sin
aprendizaje automático.
—
Un modelo que no supere claramente a la heurística manual no justifica el costo de construir, desplegar y
mantener una solución de machine learning. Esa comparación, y no la exactitud absoluta, es el verdadero
criterio de éxito del proyecto.

8. ¿Cuáles son los problemas parecidos? ¿Se pueden reutilizar
experiencias o herramientas?
Problemas análogos
Diagnóstico binario a partir de datos tabulares clínicos: predicción de diabetes
(Pima Indians), diagnóstico de cáncer de mama (Wisconsin), predicción de
enfermedad renal crónica. Comparten estructura, tamaño reducido y la misma
asimetría de costos.
Tamizaje / triaje en general: cualquier problema donde un modelo barato filtra
candidatos para una prueba costosa —detección de fraude previa a investigación
manual, control de calidad previo a inspección destructiva—. La lógica de
optimización del umbral es idéntica.
Estimación de riesgo con requisito de interpretabilidad: scoring crediticio. Ha
desarrollado técnicas de explicabilidad y de auditoría de sesgos directamente
trasladables.
Herramientas y experiencias reutilizables
Literatura específica sobre este conjunto de datos. El dataset de Cleveland es
uno de los más estudiados del repositorio UCI; existen cientos de publicaciones con
resultados de referencia, lo que permite detectar de inmediato si un resultado propio
es anómalo —tanto por debajo como, sospechosamente, por encima—.
Stack técnico estándar: scikit-learn para modelado y validación, pandas para
preparación, SHAP para explicabilidad local, imbalanced-learn si el desbalance
requiriera tratamiento (aquí no lo requiere).
Estructura del propio proyecto: la plantilla por capas ( 01_raw → 08_reporting ) y el
flujo de pull request con revisión y CI/CD ya establecidos en el repositorio son
reutilizables sin modificación.
Conocimiento clínico previo: la relación entre las variables y el desenlace está
descrita en la literatura cardiológica. Esto permite validar el modelo por coherencia:
si un modelo aprende que el dolor torácico asintomático reduce el riesgo, hay un
error en los datos o en el código.

9. ¿Hay experiencia del problema disponible?
Sí, y en tres formas distintas que conviene no confundir:
Experiencia clínica formalizada. Las guías de práctica clínica sobre dolor torácico
y cardiopatía isquémica describen el proceso diagnóstico que el modelo pretende
apoyar. El conocimiento de dominio no debe reconstruirse desde los datos.
Experiencia empírica en el conjunto de datos. Los patrones esperados por la
literatura se reproducen en el archivo entregado, lo que constituye una primera
validación de que los datos son coherentes:
Variable Categoría / condición Tasa de enfermedad n
chest_pain asymptomatic 73,0 % 1.358
nontypical 17,5 % 464
thal reversable 76,1 % 1.122
normal 22,3 % 1.581
exang angina inducida por ejercicio = sí 76,4 % 941
sex Male 55,5 % 1.944
Female 26,2 % 926
Las correlaciones puntuales con la variable objetivo ordenan las señales de la siguiente
manera: número de vasos principales coloreados por fluoroscopia ca (0,457), angina
inducida por ejercicio exang (0,432), depresión del segmento ST old_peak (0,426),
frecuencia cardíaca máxima max_hr (−0,422) y pendiente del segmento ST slope
(0,342). En el extremo opuesto, la glucemia en ayunas fbs muestra una correlación de
0,019, es decir, prácticamente nula.
Experiencia sobre el rendimiento alcanzable. Es sabido que este conjunto se
satura alrededor del 82–86 % de exactitud y de un AUC cercano a 0,90 con métodos
estándar. Los reportes que exceden ampliamente ese rango suelen deberse a fuga de
información o a evaluación sobre datos duplicados, riesgo especialmente relevante
en este archivo

10. (Importante) ¿Cómo se puede resolver el problema manualmente?
Esta pregunta es la más informativa del ejercicio, porque su respuesta define el listón
que el modelo debe superar. El problema ya se resuelve manualmente de dos maneras.
10.1 Resolución manual en la práctica clínica
Un cardiólogo revisa el caso e integra: edad y sexo, carácter del dolor torácico,
comportamiento del ECG en reposo y en esfuerzo (depresión del ST, pendiente del ST,
frecuencia cardíaca máxima alcanzada, aparición de angina), resultado de la
gammagrafía con talio y factores de riesgo metabólicos. Con esa integración estima una
probabilidad pre-test y decide la derivación. El proceso toma minutos y no requiere
ninguna herramienta computacional.
10.2 Regla heurística reproducible en una hoja de cálculo
Es posible codificar ese juicio en un sistema de puntos que cualquier persona puede
aplicar sin programación:
Condición Puntos
chest_pain = asymptomatic +2
thal ∈ {reversable, fixed} +2
ca ≥ 1 (uno o más vasos coloreados) +2
exang = 1 (angina con el ejercicio) +1
old_peak ≥ 1,5 +1
slope ∈ {2, 3} (ST plano o descendente) +1
max_hr < 140 lpm +1
sex = Male +1
age ≥ 55 años +1
Puntaje total posible 12
Evaluada sobre los 297 registros únicos y completos del archivo, esta regla rinde:
Umbral Exactitud Sensibilidad Precisión Perfil de uso
puntaje ≥ 4 77,8 % 94,9 % 68,8 % Tamizaje amplio
puntaje ≥ 5 83,5 % 89,1 % 78,2 % Recomendado (equilibrio para triaje)
puntaje ≥ 7 84,2 % 73,0 % 90,9 % Máxima exactitud, sensibilidad insuficiente
El puntaje continuo alcanza un AUC de 0,922 sobre esos mismos registros. Debe
advertirse que tanto los pesos como el umbral se eligieron observando el conjunto
completo, por lo que estas cifras son optimistas y no equivalen a una estimación
validada; su función es dimensionar el orden de magnitud del listón, no reportar un
resultado.

11. Listado de supuestos vigentes hasta este momento
Los siguientes supuestos se declaran de forma explícita para poder ser verificados o
refutados en las fases posteriores del proyecto. Ninguno debe darse por confirmado.
Sobre la definición del problema
La variable disease codifica presencia (1) frente a ausencia (0) de estenosis
coronaria significativa, y esta dicotomía es clínicamente suficiente para la decisión
de derivación.
El costo de un falso negativo es sustancialmente mayor que el de un falso positivo,
sin que se disponga de una cuantificación económica formal de esa razón.
El sistema operará como apoyo a la decisión y nunca de forma autónoma.
Sobre los datos
Las 13 variables predictoras estarán disponibles en el momento de la inferencia. El
supuesto es frágil para ca y thal , que provienen de fluoroscopia y gammagrafía —
estudios que no todo paciente tiene realizados—; conviene evaluar un modelo
alternativo que prescinda de ambas.
La etiqueta angiográfica es confiable y libre de error de medición apreciable.
Los registros son independientes entre sí: un paciente, una fila.
Las unidades son las convencionales: age en años, rest_bp en mmHg, chol en mg/
dl, max_hr en lpm, old_peak en mm de depresión del ST.
Los valores ausentes lo son de manera aproximadamente aleatoria y no informativa.
Este supuesto debe verificarse: si la ausencia de ca o thal depende de la gravedad
del paciente, la imputación introduciría sesgo.
La replicación de registros presente en el archivo es un artefacto de preparación y
no información real, por lo que debe eliminarse antes de particionar los datos.
Sobre el modelado y la población
La población de origen es representativa de aquella sobre la que se aplicará el
modelo. Este supuesto es débil y probablemente falso en un contexto
contemporáneo distinto: la muestra corresponde a pacientes derivados a angiografía
a finales de los años ochenta, con una prevalencia (46 %) muy superior a la de la
población general.
Las relaciones entre variables y desenlace son estables en el tiempo, pese a que los
criterios diagnósticos y el manejo de los factores de riesgo han cambiado desde
entonces.
El tamaño muestral efectivo —297 registros únicos— es suficiente para ajustar un
modelo interpretable con pocos parámetros, pero no para modelos de alta capacidad
ni para una partición hold-out convencional; se requiere validación cruzada repetida.
Las diferencias por sexo observadas en la tasa de enfermedad (55,5 % en hombres
frente a 26,2 % en mujeres) reflejan un patrón epidemiológico real y no un sesgo de
selección en la derivación a angiografía. Este supuesto debe examinarse antes de
cualquier despliegue, por su implicación directa en equidad.

12. ¿Cuál es la fuente de los datos?
El archivo corazon.csv corresponde a la base de datos de Cleveland del conjunto Heart
Disease del UCI Machine Learning Repository (dataset n.º 45), donado el 30 de junio de
1988.
Atributo Detalle
Institución de
origen
Cleveland Clinic Foundation. El conjunto completo integra además el Hungarian
Institute of Cardiology (Budapest), el University Hospital de Zúrich, el University
Hospital de Basilea y el V.A. Medical Center de Long Beach.
Investigadores Robert Detrano (Cleveland y V.A. Long Beach), Andras Janosi (Budapest), William
Steinbrunn (Zúrich), Matthias Pfisterer (Basilea).
Variables
originales
76 atributos por paciente; la práctica establecida —y el archivo entregado— utiliza un
subconjunto de 14.
Etiqueta original num , con valores de 0 a 4 según el número de vasos afectados. En este archivo
aparece ya dicotomizada como disease .
Modo de
obtención
Registro clínico retrospectivo de pacientes remitidos a angiografía coronaria.
Correspondencia con el archivo entregado
El archivo contiene 3.030 filas, pero al normalizar los tipos y eliminar duplicados
quedan 297 registros únicos completos —cifra que coincide exactamente con la de
la base de Cleveland tras descartar los 6 registros con valores faltantes de los 303
originales—. Esto confirma la procedencia y revela que el archivo del repositorio es una
versión replicada (aproximadamente diez veces) con valores nulos y valores
inválidos introducidos deliberadamente, presumiblemente como ejercicio de
preparación de datos.
Consecuencia metodológica que no puede omitirse. Si el conjunto se particiona en
entrenamiento y prueba sin eliminar previamente los duplicados, copias del mismo
paciente quedarán a ambos lados de la partición. El resultado será una fuga de
información que inflará artificialmente las métricas —hasta valores cercanos al 100 %— sin
que el modelo haya aprendido nada generalizable. La deduplicación debe ejecutarse
antes de cualquier partición.
Nota adicional: el repositorio incluye el archivo data/01_raw/datos_corazon_Info.txt , no analizado
en este documento. Debe consultarse para confirmar el diccionario de datos oficial del curso y
contrastarlo con la interpretación aquí adoptada.

13. ¿Cómo se actualizan los datos?
No se actualizan. Se trata de un conjunto histórico congelado: una fotografía única
de una cohorte cerrada, publicada en 1988 y sin mecanismo de actualización desde
entonces. No existe proceso de ingesta, ni fuente en vivo, ni flujo de datos nuevos hacia
el archivo. La única forma de «actualización» posible es el reemplazo manual del
archivo en data/01_raw/ por otra versión, que en el proyecto se gestiona por control de
versiones (Git y su flujo de pull request), no por un proceso automatizado de datos.
Esto tiene tres implicaciones para el diseño:
No se requiere infraestructura de ingesta ni orquestación de cargas. El
pipeline puede asumir una fuente estática.
No hay riesgo de deriva de datos durante el proyecto, pero sí una deriva ya
consumada y considerable respecto a cualquier población actual: la práctica
cardiológica ha cambiado de forma sustancial desde 1988.
El modelo resultante no es desplegable en producción clínica sin una
revalidación previa sobre datos contemporáneos y locales. El proyecto debe
entenderse como un ejercicio metodológico sobre un conjunto de referencia.
Si el sistema llegara a implantarse en una institución real, el diseño del flujo de
actualización sería el siguiente: captura desde la historia clínica electrónica en el
momento de la consulta, y confirmación de la etiqueta cuando el resultado de la
angiografía queda disponible —con una latencia natural de días a semanas entre la
predicción y la verificación de su acierto—.

14. ¿Cada cuánto tiempo se actualizan los datos?
La frecuencia de actualización de la fuente es, en sentido estricto, nula: el conjunto no
se ha modificado desde su donación en 1988. Para efectos del proyecto, la frecuencia
relevante es la de los eventos que se controlan:
Elemento Frecuencia Mecanismo
Datos crudos de la
fuente
Nunca (conjunto congelado) —
Archivo en el
repositorio
Eventual, manual Pull request con revisión y verificaciones de
CI/CD
Capas derivadas ( 02 a
05 )
En cada ejecución del pipeline Regeneradas por código a partir de
01_raw
Reentrenamiento del
modelo
Cuando cambien los datos o las
características
No hay razón para un calendario periódico
si la fuente es estática
En un despliegue real, y a título de referencia, una cadencia razonable sería:
acumulación continua de casos, reentrenamiento programado semestral o anual
(limitado por el tiempo que tarda en confirmarse la etiqueta angiográfica), y monitoreo
mensual de deriva en la distribución de las variables de entrada. Nada de esto aplica al
conjunto actual, pero conviene dejarlo consignado para no confundir el ejercicio
académico con un sistema en producción.

Referencias
Janosi, A., Steinbrunn, W., Pfisterer, M. y Detrano, R. (1988). Heart Disease [conjunto de datos]. UCI
Machine Learning Repository. https://doi.org/10.24432/C52P4X — https://archive.ics.uci.edu/dataset/
45/heart+disease
Análisis propio sobre data/01_raw/corazon.csv (3.030 filas × 14 columnas). Estadísticos, tasas
por categoría, correlaciones y evaluación de la regla heurística calculados con pandas y scikitlearn
sobre los 297 registros únicos y completos del archivo.